# Checkpoint I · Examen blanc — notebook de l'examen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_1/02_examen_notebook.ipynb)

Ce notebook accompagne le sujet `01_examen_sujet.md`.

- **Partie A, pendant l'examen** : les deux questions de code, CP1.3 (🐛, 1 point) et CP1.5 (🔨, 1,5 point). Écris ton code dans les cellules à compléter, sans `mylearn`, sans les corrigés et sans assistant IA ; `help()` et la documentation officielle de NumPy sont permis. Rien n'est vérifié pendant l'examen : les cellules « essai » montrent seulement ce que fait ton code. Tes réponses chiffrées et tes explications vont sur ta feuille (`04_mes_reponses.md`).
- **Partie B, après l'examen** : la vérification automatique de ton code et des réponses de ta feuille. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`.

« Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳ (partie A) ou ⏸️ (partie B).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_1/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP1`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Partie A · Pendant l'examen : les deux questions de code

In [ ]:
# Data of the two code questions (run this cell first)
import numpy as np

penguins = wb.datasets.load_penguins(dropna=True)          # 333 penguins without missing values
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = penguins[MEASURES].to_numpy(dtype=float)               # shape (333, 4): one row per penguin
gentoo_mass = penguins.loc[penguins["species"] == "Gentoo", "body_mass_g"].to_numpy(dtype=float)
print("X:", X.shape, "· gentoo_mass:", gentoo_mass.shape)

### CP1.3 — Partie 0 : la moyenne qui oublie `axis` 🐛 ★ ⏱️ 5 min · 1 point
**Objectif :** diagnostiquer et corriger une standardisation qui mélange les colonnes.

Un collègue veut standardiser les quatre mesures des manchots **colonne par colonne** : dans chaque colonne, retirer la moyenne de la colonne puis diviser par son écart-type (ddof = 0, comme `np.std`). Exécute son code (cellule suivante) : les moyennes des colonnes standardisées devraient valoir 0, et leurs écarts-types 1.

a) Que calcule `X.mean()` ? Quelle est la forme (*shape*) de son résultat ? Pourquoi la standardisation du collègue est-elle fausse ? (réponds sur ta feuille) **[0,3 pt]**
b) Écris `standardize(X)`, qui renvoie la version standardisée de `X` colonne par colonne (ddof = 0), sans modifier `X`. **[0,4 pt]**
c) Avec ta fonction, donne le z-score de la masse du premier manchot (ligne 0, colonne `body_mass_g`), à 2 décimales (sur ta feuille). **[0,3 pt]**

In [ ]:
# The colleague's code (do not change this cell: run it and read what it prints)
X_std_bug = (X - X.mean()) / X.std()
print("means of the columns:", X_std_bug.mean(axis=0).round(2))
print("stds of the columns :", X_std_bug.std(axis=0).round(3))

In [ ]:
def standardize(X):
    """Return the z-scores of X, column by column: (X - column mean) / column std, with ddof = 0.

    X is a 2-D array of shape (n_samples, n_features); it must not be modified.
    """
    raise NotImplementedError  # TODO CP1.3 b)

In [ ]:
# Try your function (no check during the exam: compare with what you expect)
with wb.attempt("CP1.3"):
    result = standardize(X)
    if result is None:
        print("⚠️ standardize renvoie None : as-tu oublié le return ?")
    else:
        Z_try = np.asarray(result, dtype=float)          # a DataFrame or a list is looked at as an array
        print("means of the columns:", Z_try.mean(axis=0).round(2) + 0.0)   # + 0.0 turns -0.0 into 0.0
        print("stds of the columns (np.std, ddof = 0):", Z_try.std(axis=0).round(3))
        print("z-score of the mass of the first penguin:", Z_try[0, 3])

### CP1.5 — Coder un intervalle de confiance bootstrap 🔨 ★★ ⏱️ 10 min · 1,5 point
**Objectif :** programmer le bootstrap percentile de la moyenne, sans librairie.

a) Écris `bootstrap_ci_mean(x, confidence=0.95, n_boot=1000, seed=0)`, qui renvoie l'intervalle de confiance bootstrap **percentile** de la moyenne de `x`, sous la forme d'un tuple de deux `float` `(bas, haut)`, bornes non arrondies : **[0,8 pt]**
- un seul générateur, créé dans la fonction : `rng = np.random.default_rng(seed)` ;
- pour chacun des `n_boot` rééchantillons, **dans l'ordre**, tire `n` indices avec remise, `idx = rng.integers(0, n, size=n)` ($n$ = taille de `x`), et calcule la moyenne de `x[idx]` (tirer toutes les lignes d'un coup, `size=(n_boot, n)`, ou utiliser `rng.choice(x, size=n)` donne exactement les mêmes nombres) ;
- les bornes sont les percentiles $50\,(1 - c)$ et $50\,(1 + c)$ de ces moyennes, avec $c$ = `confidence` (`np.percentile`).

Puis, sur la masse des manchots Gentoo (`gentoo_mass`, en grammes), et sur ta feuille :
b) l'intervalle à 95 % (graine 0), à 1 décimale ; **[0,2 pt]**
c) l'intervalle à 90 % (graine 0) : est-il plus large ou plus étroit ? Pourquoi ? **[0,2 pt]**
d) Laquelle de ces deux phrases est juste, et qu'est-ce qui ne va pas dans l'autre ? (1) « 95 % des manchots Gentoo pèsent entre … et … g. » (2) « Cette méthode donne un intervalle qui contient la vraie masse moyenne des Gentoo environ 95 fois sur 100. » **[0,3 pt]**

In [ ]:
def bootstrap_ci_mean(x, confidence=0.95, n_boot=1000, seed=0):
    """Percentile bootstrap confidence interval of the mean of x, as a tuple (low, high) of floats."""
    raise NotImplementedError  # TODO CP1.5 a)

In [ ]:
# Try your function (no check during the exam)
with wb.attempt("CP1.5"):
    print("95 % interval:", bootstrap_ci_mean(gentoo_mass))
    print("90 % interval:", bootstrap_ci_mean(gentoo_mass, confidence=0.90))

---

### 🛑 Fin de la partie A

L'examen s'arrête ici. Rends ta copie (enregistre ce notebook et ta feuille), puis, seulement après, passe à la partie B.

## Partie B · Après l'examen : vérification automatique

**Ne commence cette partie qu'une fois ta copie terminée**, examen rendu. Elle vérifie le code de la partie A, puis les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Elle ne note pas : les démarches, les justifications et les questions rédigées se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Exécute la vérification du code (CP1.3 et CP1.5).
3. Reporte ensuite chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). En Python, le séparateur décimal est un **point** (`0.125`).

Si tu as fermé le notebook depuis l'examen, exécute d'abord la cellule de configuration (tout en haut) et les cellules de la partie A, sauf les essais.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells of part B

RESULTS = {}        # sub-ID -> result of wb.check
PROPERTIES = {}     # question -> [(property, passed)], reset each time its check cell runs
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans la première cellule de la partie B.")
        _REMINDED.append(True)
        return False
    if "wb" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True


def ready(question, *names):
    """True when the cells of part A that define `names` have run (after a kernel restart, run them again)."""
    missing = [name for name in names if name not in globals()]
    if missing:
        verb = "n'existe" if len(missing) == 1 else "n'existent"
        print(f"⚠️ {question} : exécute d'abord les cellules de la partie A ({', '.join(missing)} {verb} pas encore).")
    return not missing


def as_letters(value):
    """Letters typed as "ABCD", "A, B, C, D" or ["A", "B", "C", "D"] -> "ABCD" (`...` stays `...`)."""
    if value is ... or value is None:
        return value
    if isinstance(value, (list, tuple)):
        value = "".join(str(v) for v in value)
    return "".join(ch for ch in str(value) if ch.isalpha()).upper()


def verdict(question, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message, and remember the result."""
    PROPERTIES.setdefault(question, []).append((success or failure, bool(ok)))
    print(f"✅ {question} : {success}" if ok else f"❌ {question} : {failure}")


_LIBRARY_ADVICE = " ; si elle fait partie de `mylearn`, ses tests (`pytest`) disent quel cas pose problème."


def code_check(ex_id, value, reminder=""):
    """wb.check of a value computed by your exam function (no advice about mylearn: it is not used here)."""
    result = wb.check(ex_id, value, computed=True, quiet=True)
    message = result.message.replace(_LIBRARY_ADVICE, ".")
    if not result and reminder and not message.startswith("Erreur classique"):
        message += f" Rappel : {reminder}"
    print(f"{'✅' if result else '❌'} Ex {ex_id} : {message}")
    RESULTS[ex_id] = result
    return result

### Le code de la partie A

In [ ]:
# Checks of standardize (CP1.3)
if exam_over() and ready("CP1.3", "penguins", "MEASURES", "standardize"):
    PROPERTIES["CP1.3"] = []
    RESULTS.pop("CP1.3c", None)
    X_fresh = penguins[MEASURES].to_numpy(dtype=float)     # a fresh copy: the trial of part A may have changed X
    X_before = X_fresh.copy()
    try:
        result = standardize(X_fresh)
    except NotImplementedError:
        print("⏳ CP1.3 : standardize n'est pas encore écrite.")
    except Exception as exc:                   # a bug of the exam code must not stop part B
        verdict("CP1.3", False, "", f"standardize(X) lève une erreur : {type(exc).__name__}: {exc}")
    else:
        verdict("CP1.3", np.array_equal(X_fresh, X_before), "X n'est pas modifié.",
                "ta fonction modifie X : calcule un nouveau tableau au lieu d'écrire dans X (pas de -= ni de /=).")
        Z_check = np.asarray(result if result is not None else np.nan, dtype=float)
        verdict("CP1.3", Z_check.shape == X_fresh.shape, "la forme est (333, 4).",
                "standardize ne renvoie rien : as-tu oublié le return ?" if result is None else
                f"le résultat doit avoir la forme de X, {X_fresh.shape}, pas {Z_check.shape}.")
        if Z_check.shape == X_fresh.shape:
            verdict("CP1.3", np.allclose(Z_check.mean(axis=0), 0, atol=1e-6), "chaque colonne a une moyenne nulle.",
                    "les moyennes des colonnes ne sont pas nulles : retire la moyenne de CHAQUE colonne (axis=0).")
            verdict("CP1.3", np.allclose(Z_check.std(axis=0), 1, atol=1e-6), "chaque colonne a un écart-type de 1 (ddof = 0).",
                    "les écarts-types des colonnes ne valent pas 1 avec ddof = 0 : divise par l'écart-type de chaque "
                    "colonne, calculé avec ddof = 0.")
            code_check("CP1.3c", float(Z_check[0, 3]),
                       "z = (x − moyenne de la colonne) / écart-type de la colonne (ddof = 0), ici pour la ligne 0 "
                       "et la colonne 3 (body_mass_g).")

In [ ]:
# Checks of bootstrap_ci_mean (CP1.5)
if exam_over() and ready("CP1.5", "gentoo_mass", "bootstrap_ci_mean"):
    PROPERTIES["CP1.5"] = []
    RESULTS.pop("CP1.5b", None)
    RESULTS.pop("CP1.5c", None)
    try:
        raw = [bootstrap_ci_mean(gentoo_mass), bootstrap_ci_mean(gentoo_mass),
               bootstrap_ci_mean(gentoo_mass, confidence=0.90)]
    except NotImplementedError:
        print("⏳ CP1.5 : bootstrap_ci_mean n'est pas encore écrite.")
    except Exception as exc:                   # a bug of the exam code must not stop part B
        verdict("CP1.5", False, "", f"bootstrap_ci_mean(gentoo_mass) lève une erreur : {type(exc).__name__}: {exc}")
    else:
        if any(ci is None for ci in raw):
            verdict("CP1.5", False, "", "bootstrap_ci_mean ne renvoie rien : as-tu oublié le return ?")
        else:
            verdict("CP1.5", all(isinstance(ci, tuple) for ci in raw), "la fonction renvoie un tuple (bas, haut).",
                    f"la fonction doit renvoyer un tuple, pas un objet de type {type(raw[0]).__name__} : "
                    "return float(low), float(high) (le reste est vérifié quand même).")
        try:
            ci_a, ci_b, ci_c = (tuple(float(v) for v in np.ravel(ci)) for ci in raw)
        except (TypeError, ValueError):
            ci_a = ci_b = ci_c = ()
        if not all(len(ci) == 2 for ci in (ci_a, ci_b, ci_c)):
            if all(ci is not None for ci in raw):
                verdict("CP1.5", False, "", "la fonction doit renvoyer deux nombres, la borne basse puis la borne haute.")
        else:
            verdict("CP1.5", ci_a == ci_b, "même graine, même intervalle.",
                    "deux appels avec la même graine donnent deux intervalles différents : crée le générateur "
                    "DANS la fonction, avec np.random.default_rng(seed).")
            if ci_a[0] == ci_a[1]:
                verdict("CP1.5", False, "", "l'intervalle a une largeur nulle : tous tes rééchantillons ont la même "
                        "moyenne. Tire les indices AVEC remise (rng.integers), avec un générateur créé une seule "
                        "fois, avant la boucle.")
            else:
                verdict("CP1.5", ci_a[0] < gentoo_mass.mean() < ci_a[1],
                        "l'intervalle contient la moyenne de l'échantillon.",
                        "l'intervalle devrait entourer la moyenne de l'échantillon : vérifie les percentiles.")
                verdict("CP1.5", ci_a[0] < ci_c[0] < ci_c[1] < ci_a[1],
                        "l'intervalle à 90 % est inclus dans celui à 95 %.",
                        "l'intervalle à 90 % devrait être inclus dans celui à 95 % : vérifie les percentiles "
                        "50 (1 − c) et 50 (1 + c).")
            checks = [code_check("CP1.5b", list(ci_a)), code_check("CP1.5c", list(ci_c))]
            if not all(checks) and not any(r.message.startswith("Erreur classique") for r in checks):
                print("   Rappel de la procédure imposée : un générateur np.random.default_rng(seed) créé dans la "
                      "fonction ; pour chacun des n_boot rééchantillons, dans l'ordre, n indices "
                      "rng.integers(0, n, size=n) et la moyenne de x[idx] ; puis les percentiles 50 (1 − c) et "
                      "50 (1 + c) de ces moyennes, sans arrondir.")

### Les réponses de ta feuille

**CP1.1 — Questions flash (le verdict seulement ; les justifications se notent avec le corrigé)**

In [ ]:
# CP1.1: the values written on your answer sheet
answer_CP1_1a = ...  # a) True / False (ou "vrai" / "faux")
answer_CP1_1b = ...  # b) True / False (ou "vrai" / "faux")
answer_CP1_1c = ...  # c) True / False (ou "vrai" / "faux")
answer_CP1_1d = ...  # d) True / False (ou "vrai" / "faux")
answer_CP1_1e = ...  # e) True / False (ou "vrai" / "faux")
answer_CP1_1f = ...  # f) True / False (ou "vrai" / "faux")

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP1_1a, answer_CP1_1b, answer_CP1_1c, answer_CP1_1d, answer_CP1_1e, answer_CP1_1f]):
        RESULTS[f"CP1.1{letter}"] = wb.check(f"CP1.1{letter}", answer)

**CP1.2 — Norme, produit scalaire et log₂**

In [ ]:
# CP1.2: the values written on your answer sheet
answer_CP1_2a = ...  # a) the norm of u
answer_CP1_2b = ...  # b) u · v
answer_CP1_2c = ...  # c) u and w orthogonal? True / False (ou "vrai" / "faux")
answer_CP1_2d = ...  # d) log2(1/32)
answer_CP1_2e = ...  # e) log2(24), 3 decimals

if exam_over():
    for letter, answer in zip("abcde", [answer_CP1_2a, answer_CP1_2b, answer_CP1_2c, answer_CP1_2d, answer_CP1_2e]):
        RESULTS[f"CP1.2{letter}"] = wb.check(f"CP1.2{letter}", answer)

**CP1.4 — Statistiques de cinq points à la main**

In [ ]:
# CP1.4: the values written on your answer sheet
answer_CP1_4a = ...  # a) [mean of x, mean of y]
answer_CP1_4b = ...  # b) [variance of x, std of x], 2 decimals
answer_CP1_4c = ...  # c) z-score of the value x = 10, 2 decimals
answer_CP1_4d = ...  # d) [covariance with ddof = 0, with ddof = 1]
answer_CP1_4e = ...  # e) correlation r, 2 decimals
answer_CP1_4f = ...  # f) [new mean of y, new median of y]

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP1_4a, answer_CP1_4b, answer_CP1_4c, answer_CP1_4d, answer_CP1_4e, answer_CP1_4f]):
        RESULTS[f"CP1.4{letter}"] = wb.check(f"CP1.4{letter}", answer)

**CP1.6 — Dépistage : matrice de confusion, precision, NPV et règle de Bayes**

In [ ]:
# CP1.6: the values written on your answer sheet
answer_CP1_6a = ...  # a) [TP, FN, FP, TN]
answer_CP1_6b = ...  # b) precision, 3 decimals
answer_CP1_6c = ...  # c) NPV, 4 decimals
answer_CP1_6d = ...  # d) accuracy, 3 decimals
answer_CP1_6f = ...  # f) P(sick | two positive tests), 3 decimals

if exam_over():
    for letter, answer in zip("abcdf", [answer_CP1_6a, answer_CP1_6b, answer_CP1_6c, answer_CP1_6d, answer_CP1_6f]):
        RESULTS[f"CP1.6{letter}"] = wb.check(f"CP1.6{letter}", answer)

**CP1.7 — Lire une courbe ROC et une courbe PR (seule la ligne de base se vérifie ici : les lectures se notent avec le corrigé)**

In [ ]:
# CP1.7: the values written on your answer sheet
answer_CP1_7e = ...  # e) height of the random 'curve' with 1 % of frauds, 2 decimals

if exam_over():
    for letter, answer in zip("e", [answer_CP1_7e]):
        RESULTS[f"CP1.7{letter}"] = wb.check(f"CP1.7{letter}", answer)

**CP1.8 — Trois hypothèses, deux lancers**

In [ ]:
# CP1.8: the values written on your answer sheet
answer_CP1_8a = ...  # a) posterior after the first heads, 2 decimals
answer_CP1_8b = ...  # b) posterior after heads, heads, 3 decimals
answer_CP1_8c = ...  # c) P(heads, heads), 3 decimals
answer_CP1_8d = ...  # d) the MAP theta (0.2, 0.5 or 0.8)
answer_CP1_8e = ...  # e) P(third throw = heads), 3 decimals

if exam_over():
    for letter, answer in zip("abcde", [answer_CP1_8a, answer_CP1_8b, answer_CP1_8c, answer_CP1_8d, answer_CP1_8e]):
        RESULTS[f"CP1.8{letter}"] = wb.check(f"CP1.8{letter}", answer)

**CP1.9 — Gradient, point selle et deux pas de descente**

In [ ]:
# CP1.9: the values written on your answer sheet
answer_CP1_9b = ...  # b) [x, y] of the critical point
answer_CP1_9d = ...  # d) [[x1, y1], [x2, y2]], exact values
answer_CP1_9e = ...  # e) [f at the start, after one step, after two steps], 3 decimals

if exam_over():
    for letter, answer in zip("bde", [answer_CP1_9b, answer_CP1_9d, answer_CP1_9e]):
        RESULTS[f"CP1.9{letter}"] = wb.check(f"CP1.9{letter}", answer)

**CP1.10 — Prédire l'effet du learning rate**

In [ ]:
# CP1.10: the values written on your answer sheet
answer_CP1_10a = ...  # a) your four letters, in the order of the learning rates (for example "AAAA")
answer_CP1_10b = ...  # b) upper bound of the learning rates that converge, 3 decimals

if exam_over():
    for letter, answer in zip("ab", [as_letters(answer_CP1_10a), answer_CP1_10b]):
        RESULTS[f"CP1.10{letter}"] = wb.check(f"CP1.10{letter}", answer)

**CP1.11 — Entropie, cross-entropy, KL et Huffman**

In [ ]:
# CP1.11: the values written on your answer sheet
answer_CP1_11a = ...  # a) H(p) in bits, 3 decimals
answer_CP1_11b = ...  # b) Huffman lengths [V, F, M, R]
answer_CP1_11c = ...  # c) mean length, 2 decimals
answer_CP1_11e = ...  # e) H(p, q) in bits, 3 decimals
answer_CP1_11f = ...  # f) KL(p || q), 3 decimals
answer_CP1_11g = ...  # g) KL(q || p), 3 decimals
answer_CP1_11h = ...  # h) is H(p, q') finite? True / False (ou "vrai" / "faux")

if exam_over():
    for letter, answer in zip("abcefgh", [answer_CP1_11a, answer_CP1_11b, answer_CP1_11c, answer_CP1_11e, answer_CP1_11f, answer_CP1_11g, answer_CP1_11h]):
        RESULTS[f"CP1.11{letter}"] = wb.check(f"CP1.11{letter}", answer)

### Bilan

In [ ]:
# Summary of the automatic checks
import re


def natural(sub_id):
    """CP1.2a before CP1.10a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP1\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    checked = [ok for props in PROPERTIES.values() for _, ok in props]
    print(f"Propriétés du code (CP1.3, CP1.5) : {sum(checked)} sur {len(checked)}.")
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; les explications (CP1.1, CP1.3 a, CP1.5 c–d, CP1.6 e et g, CP1.7 a–d et f, CP1.8 f, CP1.9 a, c et f, CP1.10 c, CP1.11 d, CP1.12 à CP1.14), la formule de CP1.10 b (seule la borne est vérifiée) ; les conclusions écrites à côté d'une valeur vérifiée (CP1.4 f, CP1.9 e, CP1.11 g et h) ; et, pour CP1.3 et CP1.5, les nombres recopiés sur ta feuille (seule ta fonction est vérifiée). Tout cela se corrige avec `03_examen_corrige.md`. Ensuite : la remédiation du corrigé (relis la synthèse, `05_synthese.md`, pour les notions ratées), puis le mini-projet.